In [76]:
import torch
from pathlib import Path

from eval import inference, top_k_accuracy
from model import LeagueDraftModel
from vocabulary import Vocabulary
from data import load_matches, ChampionDataset
from torch.utils.data import DataLoader
from draft_constraints import mask_logits

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

BATCH_SIZE = 2048

In [77]:
start = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (start, *start.parents)
    if (path / '.git').exists()
)

CHECKPOINT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'checkpoints'
DATA_DIRECTORY = PROJECT_ROOT / 'data' 

In [78]:
checkpoint = torch.load(CHECKPOINT_DIRECTORY / 'best_model.pth', device, weights_only=True)
state_dict = checkpoint['model_state_dict']
champ_dict = checkpoint['riotid_to_name']

vocab = Vocabulary(champ_dict)
model = LeagueDraftModel(len(vocab), vocab.mask_id)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoded_matches = load_matches(DATA_DIRECTORY / 'league_data.db', vocab)

test_data = ChampionDataset(encoded_matches, vocab.mask_id)

In [79]:
eval_loader = DataLoader(
    test_data, 
    batch_size = BATCH_SIZE , 
    shuffle=False,
    pin_memory=(device.type=='cuda')
)

In [80]:
print(top_k_accuracy(eval_loader, model, device, 5), top_k_accuracy(eval_loader, model, device, 10))

0.4075501752150366 0.5829882128066263


In [ ]:
# does the model use any draft context when picking? calculate top_5_accuracy with random context
count = 0
model.eval()
with torch.inference_mode(): 
    for picks, bans, target in eval_loader:

        # find where the mask id is
        masks = (picks == vocab.mask_id)

        # generate a random draft
        picks = torch.randint_like(picks, 0, len(vocab)-1, device=device)

        # set the locations that should have a mask id to mask_id
        picks[masks] = vocab.mask_id

        # keep only the  matches that have not randomly generated the target in the context
        is_valid = (picks != target.unsqueeze(1)).all(1)
        picks_final = picks[is_valid]
        bans_final = bans[is_valid]
        target_final = target[is_valid]

        picks_final = picks_final.to(device, non_blocking=(device.type == 'cuda'))
        bans_final = bans_final.to(device, non_blocking=(device.type == 'cuda'))
        target_final = target_final.to(device, non_blocking=(device.type == 'cuda'))

        logits = model(picks_final)
        logits = mask_logits(picks_final, bans_final, logits)
    
        # returns value indices pairs, only care about index
        _, preds = torch.topk(logits, k=5, dim=1)
    
        count += torch.sum(torch.sum(preds == target_final.unsqueeze(1), dim=1))

In [88]:
# top_5 accuracy with random context
print(count.item()/len(eval_loader.dataset))

0.2935223531910375


In [84]:
game = ['malphite', 'diana', 'ahri', 'masked', 'lulu', 'darius', 'warwick', 'orianna', 'ezreal', 'karma']

print(inference(model, game, vocab, device, k=5))

['Yunara', 'Jinx', 'Aphelios', 'Zeri', 'Tristana']
